In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as sm

In [7]:
campaigns = pd.read_csv('../data/northbound_campaigns.csv')
search = campaigns[campaigns['channel'] == 'Search']

In [8]:
search.tail()

,campaign_id,week_start,channel,discount_offer,spend,impressions,clicks,conversions,revenue
427,NB0428,2025-12-01,Search,0,9407.55,357833,13347,734,70266.21
431,NB0432,2025-12-08,Search,0,4691.81,173686,7544,373,35367.34
434,NB0435,2025-12-15,Search,0,11475.94,470610,17494,940,90259.37
437,NB0438,2025-12-22,Search,0,5827.12,296436,12601,574,51636.55
438,NB0439,2025-12-29,Search,0,4510.66,188606,7213,497,46921.79


In [11]:
search['impressions_k'] = search['impressions'] / 1000
model = sm.ols('conversions ~ impressions_k', data=search).fit()

In [12]:
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:            conversions   R-squared:                       0.749
Model:                            OLS   Adj. R-squared:                  0.747
Method:                 Least Squares   F-statistic:                     444.9
Date:                Fri, 06 Nov 2026   Prob (F-statistic):           1.38e-46
Time:                        14:19:48   Log-Likelihood:                -893.25
No. Observations:                 151   AIC:                             1790.
Df Residuals:                     149   BIC:                             1797.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
=================================================================================
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept         5.0398     16.835      0.299      0.765     -28.225      38.305
impressions_k     1.5406      0.073     21.092      0.000       1.396       1.685
==============================================================================
Omnibus:                        3.102   Durbin-Watson:                   1.357
Prob(Omnibus):                  0.212   Jarque-Bera (JB):                3.114
Skew:                           0.342   Prob(JB):                        0.211
Kurtosis:                       2.837   Cond. No.                         528.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [13]:
search['residuals'] = model.resid

In [16]:
worst = search.sort_values('residuals').head(10)

In [22]:
print(worst[['campaign_id', 'week_start', 'discount_offer', 'impressions_k', 'residuals']])

    campaign_id  week_start  discount_offer  impressions_k   residuals
70       NB0071  2023-06-26               0        410.157 -205.945217
159      NB0160  2024-01-22               0        297.682 -190.661431
231      NB0232  2024-07-15               0        296.732 -186.197820
276      NB0277  2024-11-04               0        473.394 -170.370839
311      NB0312  2025-02-17               0        455.427 -165.690112
34       NB0035  2023-04-03               0        300.370 -157.802679
17       NB0018  2023-02-20               0        223.244 -143.979071
212      NB0213  2024-05-27               0        272.429 -142.755581
355      NB0356  2025-06-02               0        217.020 -138.390111
18       NB0019  2023-02-27               0        215.339 -135.800290


In [23]:
best = search.sort_values('residuals', ascending=False).head(10)
print(best[['campaign_id', 'week_start', 'discount_offer', 'impressions_k', 'residuals']])

    campaign_id  week_start  discount_offer  impressions_k   residuals
341      NB0342  2025-04-28               1        126.780  228.637486
434      NB0435  2025-12-15               0        470.610  209.918310
438      NB0439  2025-12-29               0        188.606  201.385711
96       NB0097  2023-08-21               0        417.430  195.849688
142      NB0143  2023-12-04               1        224.284  181.418661
332      NB0333  2025-04-07               0        399.730  180.119064
427      NB0428  2025-12-01               0        357.833  177.667371
113      NB0114  2023-10-02               1        113.316  177.380699
391      NB0392  2025-09-08               1        251.646  168.263596
206      NB0207  2024-05-13               0        181.217  162.769520


In [24]:
model.params

Intercept        5.039835
impressions_k    1.540643
dtype: float64

In [26]:
model.params.values[0] + model.params.values[1] * 100

np.float64(159.10410330818624)

In [27]:
planned = pd.DataFrame({
    'impressions_k': [150 ,300, 450],
})
planned

,impressions_k
0,150
1,300
2,450


In [28]:
model.predict(planned)

0    236.136238
1    467.232641
2    698.329044
dtype: float64

In [ ]:
model.get_prediction(planned).summary_frame(alpha=.05) #alpha = level of confidence bound

,mean,mean_se,mean_ci_lower,mean_ci_upper,obs_ci_lower,obs_ci_upper
0,236.136238,8.459413,219.420327,252.852149,56.895232,415.377243
1,467.232641,9.991124,447.490048,486.975234,287.684078,646.781203
2,698.329044,19.188140,660.413027,736.245061,515.885791,880.772297


In [30]:
campaigns[campaigns['channel'] == 'Video'].describe()

,discount_offer,spend,impressions,clicks,conversions,revenue
count,73.000000,73.000000,7.300000e+01,73.000000,73.000000,73.000000
mean,0.315068,6509.292055,5.504132e+05,3918.712329,62.082192,5688.066164
std,0.467758,2586.208500,2.285308e+05,1700.522857,29.568073,2724.223653
min,0.000000,2103.490000,1.621300e+05,1314.000000,14.000000,1158.740000
25%,0.000000,4468.160000,3.702730e+05,2681.000000,38.000000,3922.630000
50%,0.000000,6122.170000,5.067760e+05,3582.000000,60.000000,5073.510000
75%,1.000000,7825.970000,6.733840e+05,4789.000000,82.000000,7354.550000
max,1.000000,12733.060000,1.046309e+06,9029.000000,148.000000,14462.680000


In [34]:
videos = campaigns[campaigns['channel'] == 'Video']
videos['impressions_k'] = videos['impressions'] / 1000
model_vid = sm.ols('conversions ~ impressions_k', data=videos).fit()
model_vid.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:            conversions   R-squared:                       0.677
Model:                            OLS   Adj. R-squared:                  0.672
Method:                 Least Squares   F-statistic:                     148.5
Date:                Fri, 06 Nov 2026   Prob (F-statistic):           4.49e-19
Time:                        14:59:58   Log-Likelihood:                -309.10
No. Observations:                  73   AIC:                             622.2
Df Residuals:                      71   BIC:                             626.8
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
=================================================================================
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept         3.5041      5.199      0.674      0.502      -6.862      13.870
impressions_k     0.1064      0.009     12.188      0.000       0.089       0.124
==============================================================================
Omnibus:                        2.859   Durbin-Watson:                   1.850
Prob(Omnibus):                  0.239   Jarque-Bera (JB):                2.249
Skew:                           0.216   Prob(JB):                        0.325
Kurtosis:                       3.743   Cond. No.                     1.56e+03
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.56e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [38]:
1/0.1064 # impression / sale (using video channel)

9.398496240601505

In [37]:
1/1.540643 # impression / sale (using search channel)

0.6490796375279673

In [39]:
search.head()

,campaign_id,week_start,channel,discount_offer,spend,impressions,clicks,conversions,revenue,impressions_k,residuals
1,NB0002,2023-01-09,Search,0,4512.46,198756,9592,220,20834.90,198.756,-91.251813
5,NB0006,2023-01-16,Search,0,3063.68,131636,5579,167,16614.48,131.636,-40.843875
7,NB0008,2023-01-23,Search,0,3851.73,221903,9663,256,23956.14,221.903,-90.913069
10,NB0011,2023-01-30,Search,0,5287.11,226840,9564,273,26290.07,226.840,-81.519222
12,NB0013,2023-02-06,Search,0,7094.57,226455,12134,327,30421.52,226.455,-26.926074


In [40]:
videos['spend'].sum() / videos['impressions'].sum()

np.float64(0.011826192533175761)

In [41]:
search['spend'].sum() / search['impressions'].sum()

np.float64(0.0242290763531843)

In [ ]:
#Cost to sell 1 unit
93985*0.01182 # Video

1110.9027

In [44]:
649*0.0242 #search

15.7058